# 04 – Spatial Analysis with Real Data

This notebook demonstrates how to incorporate real geographic and population
data into the simulation.  When the data files are absent it falls back to
synthetic data so you can always run it end-to-end.

**Data files expected in `data/raw/`:**
- `boundary.geojson` – study-area boundary
- `population.tif` – population density raster
- `businesses.csv` – business locations with `lon`/`lat` columns

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
DATA_DIR = pathlib.Path('../data/raw')

In [ ]:
from hotelling.spatial.map_loader import MapLoader
from hotelling.spatial.population import PopulationGrid
from hotelling.spatial.business_locations import BusinessLocations

boundary_path = DATA_DIR / 'boundary.geojson'
map_loader = MapLoader(filepath=boundary_path if boundary_path.exists() else None)

pop_path = DATA_DIR / 'population.tif'
pop_grid = PopulationGrid(
    raster_path=pop_path if pop_path.exists() else None,
    map_loader=map_loader
)

biz_path = DATA_DIR / 'businesses.csv'
biz_locs = BusinessLocations(
    filepath=biz_path if biz_path.exists() else None,
    map_loader=map_loader
)

print(f'Business locations loaded: {len(biz_locs)}')

In [ ]:
# Sample population-weighted consumer locations
consumer_locs = pop_grid.sample_locations(n=2000, seed=42)
print('Consumer locations shape:', consumer_locs.shape)

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 7))
    plt.scatter(consumer_locs[:, 0], consumer_locs[:, 1], s=3, alpha=0.2, label='Consumers')
    sampled = biz_locs.sample(n=5, seed=0)
    if sampled:
        xs, ys = zip(*sampled)
        plt.scatter(xs, ys, s=150, marker='*', color='red', label='Businesses', zorder=5)
    plt.xlabel('x'); plt.ylabel('y')
    plt.title('Spatial Distribution of Consumers and Businesses')
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install matplotlib: pip install matplotlib')